In [ ]:
import pandas as pd
import mlflow

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

import joblib
import sys
import os
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

from src.features.engineer import create_preprocessor, FeatureEngineer
from src.preprocess.preprocessor import preprocess

import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import logging

# Stop printing MLflow và alembic
logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("alembic").setLevel(logging.ERROR)

#### **1. Preparing Data**

In [ ]:
df = pd.read_csv("../data/raw/train.csv")
df = preprocess(df=df)

target = "Churn"

X = df.drop(columns=[target])
y = df[target]

print(f'X shape: {X.shape}, y shape: {y.shape}')

In [ ]:
# Step 1: Train + Temp (80% - 20%)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 2: Split train into Train + Validation (60% - 20%)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.25,  # 0.25 * 0.8 = 0.2
    random_state=42,
    stratify=y_train_full
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

#### **2. Model Pipeline**

In [ ]:
## 1. Random Forest Pipeline

# full pipeline
pipeline_random = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("preprocessor", create_preprocessor()),
    ("model", RandomForestClassifier(random_state=42))
])

# hyperparameter tuning space
param_dist_random = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [5, 10, 15, None],
    "model__min_samples_split": [2, 5, 10]
}

## 2. Logistic 

# Pipeline cho Logistic Regression
pipeline_logistic = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("preprocessor", create_preprocessor()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

# hyperparameter tuning space
param_dist_logistic = {
    "model__C": [0.1, 1, 10],          
    "model__penalty": ["l1", "l2"],      
    "model__solver": ["liblinear"]       
}

## 3. XG Boost

# High-performance classification pipeline using XGBoost
pipeline_xgb = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("preprocessor", create_preprocessor()),
    ("model", XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss' # Standard for binary classification
    ))
])

# Hyperparameter tuning space for XGBoost
param_dist_xgb = {
    "model__n_estimators": [100, 200, 500],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [3, 6, 9],
    "model__subsample": [0.7, 0.8, 0.9],
    "model__colsample_bytree": [0.7, 0.8, 0.9]
}

## 4. Gaussian Naive Bayes 

pipeline_nb = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("preprocessor", create_preprocessor()),
    ("model", GaussianNB())
])

# Hyperparameter tuning space for Naive Bayes
param_dist_nb = {
    "model__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
}


#### **3. Model Tuning and Evaluation**

In [ ]:
# List of pipelines and their corresponding parameter distributions
models_to_tune = [
    ("Random Forest", pipeline_random, param_dist_random),
    ("Logistic Regression", pipeline_logistic, param_dist_logistic),
    ("XGBoost", pipeline_xgb, param_dist_xgb),
    ("Naive Bayes", pipeline_nb, param_dist_nb)
]

# Initialize Experiment
mlflow.set_experiment("Customer_Churn_Analysis_Group04")

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

comparison_metrics = []
best_models_dict = {}
val_metrics_list = []

for name, pipeline, param_dist in models_to_tune:
    with mlflow.start_run(run_name=name):
        print(f"--- Tuning: {name} ---")
        
        # 1. Tuning on TRAIN
        search = RandomizedSearchCV(
            pipeline, 
            param_distributions=param_dist, 
            n_iter=20,
            scoring="f1", 
            cv=3, 
            n_jobs=4, 
            random_state=42,
            verbose=1
        )
        search.fit(X_train, y_train)
        
        best_model = search.best_estimator_
        best_models_dict[name] = best_model
        
        # 2. Evaluate on VALIDATION
        y_val_pred = best_model.predict(X_val)
        y_val_proba = best_model.predict_proba(X_val)[:, 1]
        
        acc_val = accuracy_score(y_val, y_val_pred)
        prec_val = precision_score(y_val, y_val_pred)
        rec_val = recall_score(y_val, y_val_pred)
        f1_val = f1_score(y_val, y_val_pred)
        f1_mac_val = f1_score(y_val, y_val_pred, average="macro")
        auc_val = roc_auc_score(y_val, y_val_proba)
        
        val_metrics = {
            "Model": name,
            "Accuracy_val": acc_val,
            "Precision_val": prec_val,
            "Recall_val": rec_val,
            "F1_val": f1_val,
            "F1_macro_val": f1_mac_val,
            "AUC_val": auc_val
        }
        
        val_metrics_list.append(val_metrics)
        
        print(f"{name} - F1_val: {f1_val:.4f} | AUC_val: {auc_val:.4f}")
        
        # Log MLflow
        mlflow.log_params(search.best_params_)
        mlflow.log_metrics({
            "accuracy_val": acc_val,
            "precision_val": prec_val,
            "recall_val": rec_val,
            "f1_val": f1_val,
            "f1_macro_val": f1_mac_val,
            "auc_val": auc_val
        })

df_val = pd.DataFrame(val_metrics_list).set_index("Model")
print("\nValidation Performance:")
print(df_val.sort_values(by="F1_val", ascending=False).to_markdown())

In [ ]:
# Confusion Matrix on Validation
n_models = len(best_models_dict)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))

for i, (name, model) in enumerate(best_models_dict.items()):
    # 1. Get predictions
    y_pred = model.predict(X_val)
    
    # 2. Compute confusion matrix
    cm = confusion_matrix(y_val, y_pred)
    
    # 3. Plot using Seaborn
    # format: [[TN, FP], [FN, TP]]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    
    axes[i].set_title(f'Confusion Matrix: {name}')
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')
    axes[i].set_xticklabels(['Stay (0)', 'Churn (1)'])
    axes[i].set_yticklabels(['Stay (0)', 'Churn (1)'])

plt.tight_layout()
plt.show()

In [ ]:
# best model selection base on Validation set
best_model_name = df_val["F1_val"].idxmax()
print("\nBest model based on Validation:", best_model_name)

#### **4. Final Evaluation on Test set**

In [ ]:
final_model = best_models_dict[best_model_name]

y_test_pred = final_model.predict(X_test)
y_test_proba = final_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_test_pred)
prec = precision_score(y_test, y_test_pred)
rec = recall_score(y_test, y_test_pred)
f1_bin = f1_score(y_test, y_test_pred)
f1_mac = f1_score(y_test, y_test_pred, average="macro")
auc = roc_auc_score(y_test, y_test_proba)

final_metrics = {
    "Model": best_model_name,
    "Accuracy": acc,
    "Precision": prec,
    "Recall": rec,
    "F1 (Binary)": f1_bin,
    "F1 (Macro)": f1_mac,
    "AUC": auc
}

print("\nFinal Test Performance:")
print(pd.DataFrame([final_metrics]).set_index("Model").to_markdown())

In [ ]:
# Feature Importance
final_model = best_models_dict[best_model_name]
# Take final model
model = final_model.named_steps["model"]

# Take Feature name after preprocessing
feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()

# Importance Feature
importances = model.feature_importances_

# DataFrame
feat_imp = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

# Plot
plt.figure(figsize=(8,5))
plt.barh(feat_imp["Feature"][:10][::-1], feat_imp["Importance"][:10][::-1])
plt.title("Top 10 Feature Importances")
plt.xlabel("Importance")
plt.show()

#### **5. Save Model**

In [ ]:
os.makedirs("../models", exist_ok=True)
path = "../models/model.pkl"

# Save model
joblib.dump(final_model, path)

# Print absolute path
print("Model saved at:", os.path.abspath(path))